# Quantile delta mapping: Comparison of two windowing approaches

**Quantile Delta Mapping (QDM)** is a bias-correction technique designed to adjust model outputs so their statistical distributions match real-world observations, all while preserving the raw model's projected long-term trend. This notebook compares two sampling approaches to calculate and adjust the model temperature output with respect to observations: A day-of-year windowing which preserves the seasonal variability, and a combined day-of-year and hour-of-day ("hour-of-year") windowing which preserves both and seasonal and diurnal variability, the latter of which has both a strong seasonal and diurnal signal for temperature. 

**Time period definitions:** IOUs have often expressed a desire to use recent observations for localization rather than being constrained to the relatively early `historical` simulations. With that in mind:

* The data used here as the `reference` (`ref`) period spans 1992--2021; since this goes beyond the historical simulation time period, it is referred to as `reference` instead of `historical`.

* The data used here as the `projection` (`proj`) period spans 1946--2075. No real reason for that, just what I picked.

**Note:** We need at least 30 years of data to achieve the statistical robustness necessary for localization.

In [ ]:
import numpy as np
import scipy.stats as stats
import xarray as xr
from xclim.core.units import convert_units_to
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import pandas as pd
import seaborn as sns
import climakitae as ck
from typing import Tuple, Dict, Union, Optional

## Details of the QDM procedures used here

### Step 1: Reshape into a 3D Calendar Grid

Before computing statistical distributions, the 1D continuous time series (spanning many years) is unstacked into a **3D grid** organized by:

* **Year** (e.g., 1980, 1981, …)
* **Day of Year** ($1 \dots 365$)
* **Hour of Day** ($0 \dots 23$)

This structure isolates seasonal and diurnal patterns across the record.

---

### Step 2: Apply Circular Boundary Padding

Weather patterns transition smoothly across time boundaries. To prevent artificial gaps at calendar edges:

* **Day-of-Year Padding:** The end of December is wrapped around to pad early January, and vice versa.
* **Hour-of-Day Padding:** Late-night hours ($23:00$) are wrapped around to pad early morning hours ($00:00$).

---

### Step 3: Pool Data into Rolling Windows

Rather than evaluating each hour or day in strict isolation (which might not contain enough historical data points for stable statistics), data is gathered using a moving window pool:

* **Day-of-Year Windowing:** A multi-day window (e.g., 61 days) centered on a target day of the year pools neighboring days across all available historical years.
* **Hour-of-Year Windowing:** A multi-hour window (e.g., 3 hours) centered on a target hour of day pools neighboring hours around the target time (e.g., $11:00 \text{ to } 15:00$) alongside the multi-day window, capturing diurnal variability with increased sample sizes.

All values falling within the configured rolling windows across all reference years are combined into a single statistical sample pool for that specific calendar point.

---

### Step 4: Compute Quantile Matrices

For every pooled slice $(d, h)$, the algorithm calculates sample quantiles across a probability grid $q \in [0.01, 0.99]$ for three datasets:

1. **$Q_{\text{obs}}$:** Observed baseline values (the ground truth target).
2. **$Q_{\text{ref}}$:** Raw historical model values (the model's baseline performance).
3. **$Q_{\text{proj}}$:** Raw future/projection model values (the raw model output to be corrected).

---

### Step 5: Quantile Delta Mapping (The Correction Core)

For every individual value in the target model time series $x_{\text{model}}$ at day $d$ and hour $h$:

1. **Find Model Rank:** Determine the percentile rank $q$ of $x_{\text{model}}$ within its historical model reference distribution $Q_{\text{ref}}$.
2. **Map to Observations:** Find the observed value $x_{\text{obs}}$ corresponding to that same percentile rank $q$ in $Q_{\text{obs}}$.
3. **Calculate Model Trend (Delta):** Compute the relative change predicted by the model for that percentile:
* **Additive Mode:** $\Delta = x_{\text{model}} - Q_{\text{ref}}(q)$
* **Multiplicative Mode:** $\Delta = \frac{x_{\text{model}}}{Q_{\text{ref}}(q)}$


4. **Combine Mapped Value & Delta:** Apply the model's projected delta to the observed value:
* **Additive Final:** $x_{\text{adjusted}} = x_{\text{obs}} + \Delta$
* **Multiplicative Final:** $x_{\text{adjusted}} = x_{\text{obs}} \times \Delta$



---

### Step 6: Flatten Back to a 1D Time Series

Finally, the adjusted 3D grid $(\text{year}, \text{dayofyear}, \text{hour})$ is stacked back down into a continuous 1D time series matching the original datetime index and metadata attributes.

In [ ]:
# =====================================================================
# QDM support functions
# =====================================================================

def _pad_dayofyear_cyclically(da_3d, window_size):
    """Pad a 3D (year, dayofyear, hour) DataArray circularly along dayofyear.

    Parameters
    ----------
    da_3d : xarray.DataArray
        Input 3D DataArray with dimensions ('year', 'dayofyear', 'hour').
    window_size : int
        Total width of the rolling day-of-year window in days.

    Returns
    -------
    xarray.DataArray
        DataArray with circular day-of-year boundary values attached to both ends.
    """
    # Calculate integer offset for half-window boundary padding
    half_win = window_size // 2

    # Extract late December slices to append before January 1
    head_pad = da_3d.isel(dayofyear=slice(-half_win, None)).copy()

    # Extract early January slices to append after December 31
    tail_pad = da_3d.isel(dayofyear=slice(0, half_win)).copy()

    # Determine boundary limits of the day-of-year coordinate array
    min_doy = int(da_3d.dayofyear.values[0])
    max_doy = int(da_3d.dayofyear.values[-1])

    # Assign negative and overflow coordinate values to align padding slices
    head_pad["dayofyear"] = np.arange(min_doy - half_win, min_doy)
    tail_pad["dayofyear"] = np.arange(max_doy + 1, max_doy + 1 + half_win)

    # Concatenate head padding, interior data, and tail padding along day-of-year
    return xr.concat([head_pad, da_3d, tail_pad], dim="dayofyear")


def _pad_hour_cyclically(da_3d, window_size):
    """Pad a 3D (year, dayofyear, hour) DataArray circularly along hour.

    Parameters
    ----------
    da_3d : xarray.DataArray
        Input 3D DataArray with dimensions ('year', 'dayofyear', 'hour').
    window_size : int
        Total width of the rolling hour window in hours.

    Returns
    -------
    xarray.DataArray
        DataArray with circular hour boundary values attached to both ends.
    """
    # Calculate integer offset for half-window boundary padding
    half_win = window_size // 2

    # Extract late-night hour slices to append before hour 0
    head_pad = da_3d.isel(hour=slice(-half_win, None)).copy()

    # Extract early-morning hour slices to append after hour 23
    tail_pad = da_3d.isel(hour=slice(0, half_win)).copy()

    # Determine boundary limits of the hour coordinate array
    min_hour = int(da_3d.hour.values[0])
    max_hour = int(da_3d.hour.values[-1])

    # Assign negative and overflow coordinate values to align padding slices
    head_pad["hour"] = np.arange(min_hour - half_win, min_hour)
    tail_pad["hour"] = np.arange(max_hour + 1, max_hour + 1 + half_win)

    # Concatenate head padding, interior data, and tail padding along hour dimension
    return xr.concat([head_pad, da_3d, tail_pad], dim="hour")


def _to_year_doy_hour_grid(ds):
    """Convert a 1D time-series DataArray into a 3D (year, doy, hour) array.

    Parameters
    ----------
    ds : xarray.DataArray or xarray.Dataset
        Input 1D time-series data indexed by datetime coordinates.

    Returns
    -------
    xarray.DataArray
        Reshaped 3D DataArray with dimensions ('year', 'dayofyear', 'hour').
    """
    # Extract underlying DataArray if input dataset is supplied
    da = ds.to_array().squeeze() if isinstance(ds, xr.Dataset) else ds

    # Extract calendar components from datetime coordinate
    year = da.time.dt.year
    doy = da.time.dt.dayofyear
    hour = da.time.dt.hour

    # Construct Pandas MultiIndex from temporal components
    index = pd.MultiIndex.from_arrays(
        [year.values, doy.values, hour.values],
        names=["year", "dayofyear", "hour"],
    )

    # Attach MultiIndex to array values and unstack into 3D grid
    da_indexed = xr.DataArray(
        da.values, coords={"index": index}, dims=["index"]
    ).unstack("index")

    return da_indexed


def _unstack_to_da(corr_3d, ds):
    """Flatten a 3D (year, dayofyear, hour) DataArray back into a 1D series.

    Parameters
    ----------
    corr_3d : xarray.DataArray
        Adjusted 3D DataArray with dimensions ('year', 'dayofyear', 'hour').
    ds : xarray.DataArray or xarray.Dataset
        Reference 1D input supplying target time coordinates and metadata attributes.

    Returns
    -------
    xarray.DataArray
        1D DataArray indexed by the original datetime coordinate.
    """
    # Extract underlying DataArray if input dataset is supplied
    da = ds.to_array().squeeze() if isinstance(ds, xr.Dataset) else ds

    # Stack 3D grid dimensions into single multi-index and remove NaN padding
    flattened = corr_3d.stack(
        time=("year", "dayofyear", "hour")
    ).dropna("time")

    # Construct 1D DataArray using original metadata attributes and time index
    return xr.DataArray(
        flattened.values,
        coords={"time": da.time.values},
        dims=["time"],
        attrs=da.attrs,
    )


def _apply_qdm(target_3d, q_ref, q_obs, q_grid, kind="additive"):
    """Apply Quantile Delta Mapping adjustment factors to a 3D dataset.

    Parameters
    ----------
    target_3d : xarray.DataArray
        Input 3D model dataset (year, dayofyear, hour) to bias-adjust.
    q_ref : xarray.DataArray
        Reference model quantile matrix.
    q_obs : xarray.DataArray
        Observed baseline quantile matrix.
    q_grid : numpy.ndarray
        1D array of quantile probability levels spanning [0, 1].
    kind : {'additive', 'multiplicative'}, default 'additive'
        Type of bias correction mode to apply.

    Returns
    -------
    xarray.DataArray
        Adjusted 3D DataArray containing bias-corrected target values.
    """
    # Initialize output array filled with NaN values matching target grid dimensions
    adjusted_3d = xr.full_like(target_3d, np.nan)
    
    # Check whether quantile matrices contain hour dimension for sub-daily mapping
    has_hour_dim = "hour" in q_ref.dims

    # Loop over day-of-year and hour dimensions to process time slices
    for doy in range(1, 366):
        for h in range(24):
            # Extract 1D array of time-series values across years for target slice
            t_vals = target_3d.sel(dayofyear=doy, hour=h).values

            # Skip processing if slice contains missing data
            if np.all(np.isnan(t_vals)):
                continue

            # Extract corresponding 1D quantile vectors for reference and observations
            if has_hour_dim:
                q_ref_vec = q_ref.sel(dayofyear=doy, hour=h).values
                q_obs_vec = q_obs.sel(dayofyear=doy, hour=h).values
            else:
                q_ref_vec = q_ref.sel(dayofyear=doy).values
                q_obs_vec = q_obs.sel(dayofyear=doy).values

            # Step 1: Interpolate percentile rank of target model values within reference distribution
            target_ranks = np.interp(t_vals, q_ref_vec, q_grid)

            # Step 2: Map estimated percentile ranks to observed baseline quantile values
            obs_mapped = np.interp(target_ranks, q_grid, q_obs_vec)

            # Step 3: Calculate model delta and apply adjustment factor
            if kind == "additive":
                # Compute additive model delta relative to reference quantile rank
                target_delta = t_vals - np.interp(
                    target_ranks, q_grid, q_ref_vec
                )
                # Assign adjusted value sum to target slice
                adjusted_3d.loc[dict(dayofyear=doy, hour=h)] = (
                    obs_mapped + target_delta
                )
            else:
                # Interpolate reference quantile value at target rank
                ref_mapped = np.interp(target_ranks, q_grid, q_ref_vec)
                # Compute multiplicative model ratio with division-by-zero protection
                target_ratio = t_vals / np.where(
                    ref_mapped == 0, 1e-6, ref_mapped
                )
                # Assign adjusted value product to target slice
                adjusted_3d.loc[dict(dayofyear=doy, hour=h)] = (
                    obs_mapped * target_ratio
                )

    return adjusted_3d


# =====================================================================
# QDM pipeline
# =====================================================================


def run_qdm_pipeline(
    ds_obs: Union[xr.DataArray, xr.Dataset],
    ds_ref: Union[xr.DataArray, xr.Dataset],
    ds_proj: Optional[Union[xr.DataArray, xr.Dataset]] = None,
    day_window: int = 91,
    hour_window: Optional[int] = 5,
    n_quantiles: int = 100,
    kind: str = "additive",
    return_intermediate: bool = True,
) -> Union[
    Tuple[xr.DataArray, xr.DataArray],
    Tuple[xr.DataArray, xr.DataArray, Dict[str, xr.DataArray]],
]:
    """Execute Quantile Delta Mapping (QDM) with circular rolling windows.

    Computes and returns bias-adjusted versions for both the baseline
    reference (`ds_ref`) and optional future projection (`ds_proj`).

    Parameters
    ----------
    ds_obs : xarray.DataArray or xarray.Dataset
        Observed historical baseline time series.
    ds_ref : xarray.DataArray or xarray.Dataset
        Model historical baseline reference time series.
    ds_proj : xarray.DataArray or xarray.Dataset, optional
        Model future projection time series. Default is copy of `ds_ref`.
    day_window : int, default 91
        Width of rolling day-of-year window for pooling quantiles.
    hour_window : int, optional, default 5
        Width of rolling hour window for pooling quantiles.
    n_quantiles : int, default 100
        Number of quantile probability levels between 0.01 and 0.99.
    kind : {'additive', 'multiplicative'}, default 'additive'
        Type of bias correction mode to apply.
    return_intermediate : bool, default True
        Whether to return intermediate quantile matrices and diagnostics.

    Returns
    -------
    ds_qdm_proj : xarray.DataArray
        QDM bias-adjusted future projection time series.
    ds_qdm_ref : xarray.DataArray
        QDM bias-adjusted historical baseline reference time series.
    diagnostics : dict of {str: xarray.DataArray}, optional
        Diagnostic outputs (e.g. quantile matrices).
    """
    # Use a copy of the reference dataset if no projection dataset is supplied
    if ds_proj is None:
        ds_proj = ds_ref.copy()

    # Define linear probability spacing for empirical quantile evaluation
    q_grid = np.linspace(0.01, 0.99, n_quantiles)
    
    # Calculate integer offset for daily window centering
    half_win_day = day_window // 2

    # Reshape 1D time-series datasets into 3D grids of shape (year, dayofyear, hour)
    obs_3d = _to_year_doy_hour_grid(ds_obs)
    ref_3d = _to_year_doy_hour_grid(ds_ref)
    proj_3d = _to_year_doy_hour_grid(ds_proj)

    # Attach cyclic boundary padding along day-of-year axis to handle year-end transitions
    obs_padded = _pad_dayofyear_cyclically(obs_3d, day_window)
    ref_padded = _pad_dayofyear_cyclically(ref_3d, day_window)
    proj_padded = _pad_dayofyear_cyclically(proj_3d, day_window)

    # Apply cyclic boundary padding along hour axis to handle midnight transitions
    if hour_window is not None and hour_window > 1:
        half_win_hour = hour_window // 2
        obs_padded = _pad_hour_cyclically(obs_padded, hour_window)
        ref_padded = _pad_hour_cyclically(ref_padded, hour_window)
        proj_padded = _pad_hour_cyclically(proj_padded, hour_window)

    # Construct rolling window pools along the day-of-year dimension
    obs_win = obs_padded.rolling(
        dayofyear=day_window, center=True
    ).construct("win_day")
    ref_win = ref_padded.rolling(
        dayofyear=day_window, center=True
    ).construct("win_day")
    proj_win = proj_padded.rolling(
        dayofyear=day_window, center=True
    ).construct("win_day")

    # Crop padded day-of-year boundary slices to leave 365 interior calendar days
    obs_win = obs_win.isel(
        dayofyear=slice(half_win_day, half_win_day + 365)
    )
    ref_win = ref_win.isel(
        dayofyear=slice(half_win_day, half_win_day + 365)
    )
    proj_win = proj_win.isel(
        dayofyear=slice(half_win_day, half_win_day + 365)
    )

    # Construct rolling window pools along the hour dimension if enabled
    if hour_window is not None and hour_window > 1:
        obs_win = obs_win.rolling(
            hour=hour_window, center=True
        ).construct("win_hour")
        ref_win = ref_win.rolling(
            hour=hour_window, center=True
        ).construct("win_hour")
        proj_win = proj_win.rolling(
            hour=hour_window, center=True
        ).construct("win_hour")

        # Crop padded hour boundary slices to leave 24 interior hours
        obs_win = obs_win.isel(
            hour=slice(half_win_hour, half_win_hour + 24)
        )
        ref_win = ref_win.isel(
            hour=slice(half_win_hour, half_win_hour + 24)
        )
        proj_win = proj_win.isel(
            hour=slice(half_win_hour, half_win_hour + 24)
        )
        # Pool across years, day windows, and hour windows for sub-daily QDM
        pool_dims = ["year", "win_day", "win_hour"]
    else:
        # Pool across years and day windows for daily QDM
        pool_dims = ["year", "win_day", "hour"]

    # Calculate quantile probability matrices across pooled dimensions
    q_obs = obs_win.quantile(
        q_grid, dim=pool_dims
    ).rename({"quantile": "q"})
    q_ref = ref_win.quantile(
        q_grid, dim=pool_dims
    ).rename({"quantile": "q"})
    q_proj = proj_win.quantile(
        q_grid, dim=pool_dims
    ).rename({"quantile": "q"})

    # Compute baseline adjustment factors based on mode
    if kind == "additive":
        delta_matrix = q_obs - q_ref
    else:
        delta_matrix = q_obs / np.where(q_ref == 0, 1e-6, q_ref)

    # Map quantile ranks and apply adjustment factors to 3D grids
    adjusted_ref_3d = _apply_qdm(ref_3d, q_ref, q_obs, q_grid, kind=kind)
    adjusted_proj_3d = _apply_qdm(proj_3d, q_ref, q_obs, q_grid, kind=kind)

    # Flatten 3D adjusted grids back into 1D time-series DataArrays
    ds_qdm_ref = _unstack_to_da(adjusted_ref_3d, ds_ref)
    ds_qdm_proj = _unstack_to_da(adjusted_proj_3d, ds_proj)

    # Assemble diagnostic metadata and intermediate quantile matrices
    diagnostics = {
        "q_grid": q_grid,
        "q_obs": q_obs,
        "q_ref": q_ref,
        "q_proj": q_proj,
        "delta_matrix": delta_matrix,
        "correction_mode": kind,
    }

    if return_intermediate:
        return ds_qdm_proj, ds_qdm_ref, diagnostics
    return ds_qdm_proj, ds_qdm_ref


# Rolling to PST from UTC is technically for the plots but is
# still a priority for our devs to review

def _roll_to_pst(matrix: np.ndarray, pst_offset: int = -8) -> np.ndarray:
    """Roll a 2D (24, 365) matrix along the hour axis to convert to PST."""
    return np.roll(matrix, pst_offset, axis=0)


def _get_monthly_diurnal_cycle_pst(
    da: xr.DataArray, gap_width: int = 2, pst_offset: int = -8
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, list]:
    """Compute the monthly mean diurnal profile in PST with NaN gaps
    to separate each month."""

    monthly_diurnal = da.groupby("time.month").map(
        lambda m_da: m_da.groupby("time.hour").mean("time")
    )
    # PST conversion here
    diurnal_pst = np.roll(monthly_diurnal.values, pst_offset, axis=1)

    series_with_gaps = []
    x_coords = []
    month_centers = []
    current_x = 0

    month_labels = [
        "Jan", "Feb", "Mar", "Apr", "May", "Jun",
        "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
    ]

    for m in range(12):
        month_vals = diurnal_pst[m, :]
        series_with_gaps.extend(month_vals)
        x_coords.extend(range(current_x, current_x + 24))

        month_centers.append(current_x + 11.5)
        current_x += 24

        if m < 11:
            series_with_gaps.extend([np.nan] * gap_width)
            x_coords.extend(range(current_x, current_x + gap_width))
            current_x += gap_width

    return (
        np.array(x_coords),
        np.array(series_with_gaps),
        np.array(month_centers),
        month_labels,
    )

# =====================================================================
# Diagnostic plotting functions
# =====================================================================


# Part 1: QDM results

# Define visual style palette
COLOR_OBS = "#000000"       # Black (Observed Baseline)
COLOR_RAW_REF = "#D55E00"   # Vermillion (Raw Historical Model Reference)
COLOR_QDM_REF = "#E69F00"   # Orange (QDM Bias-Adjusted Reference)
COLOR_RAW_PROJ = "#0072B2"  # Deep Blue (Raw Future Model Projection)
COLOR_QDM_PROJ = "#009E73"  # Bluish Green (QDM Bias-Adjusted Projection)


def _compute_hourly_mean_heatmap(
    ds: Union[xr.DataArray, xr.Dataset]
) -> np.ndarray:
    """Compute a (24, 365) hour x dayofyear mean climatology matrix directly 
    from time-series data.

    Calculates the mean value across years for each hour of each day of the year,
    to produce a smooth (24, 365) climatology.

    Parameters
    ----------
    ds : xarray.DataArray or xarray.Dataset
        Input 1D time-series data indexed by time.

    Returns
    -------
    numpy.ndarray
        2D mean climatology matrix of shape (24, 365).
    """
    # Extract underlying DataArray if input dataset is supplied
    da = ds.to_array().squeeze() if isinstance(ds, xr.Dataset) else ds

    # Extract temporal components from time coordinate
    year = da.time.dt.year
    doy = da.time.dt.dayofyear
    hour = da.time.dt.hour

    # Construct Pandas MultiIndex from temporal components
    index = pd.MultiIndex.from_arrays(
        [year.values, doy.values, hour.values],
        names=["year", "dayofyear", "hour"],
    )

    # Attach MultiIndex and unstack into 3D grid
    da_3d = xr.DataArray(
        da.values, coords={"index": index}, dims=["index"]
    ).unstack("index")

    # Calculate mean across all years for each (dayofyear, hour) coordinate
    mean_2d = da_3d.mean(dim="year")
    matrix = mean_2d.transpose("hour", "dayofyear").values

    # Slice matrix to retain 365 calendar days
    return matrix[:, :365]


def _extract_quantile_heatmap(
    diagnostics: Dict[str, xr.DataArray],
    key: str,
    eval_quantile: float = 0.50,
) -> np.ndarray:
    """Extract a (24, 365) matrix directly from QDM pipeline diagnostics.

    Parameters
    ----------
    diagnostics : dict of {str: xarray.DataArray}
        Pipeline diagnostics dictionary returned by `run_qdm_pipeline`.
    key : str
        Target diagnostic dictionary key (e.g., 'q_obs', 'q_ref', 'q_proj').
    eval_quantile : float, default 0.50
        Quantile level (0.0 to 1.0) to evaluate.

    Returns
    -------
    numpy.ndarray
        2D matrix of shape (24, 365).
    """
    # Retrieve target DataArray from diagnostics dictionary
    q_da = diagnostics[key]
    
    # Identify quantile dimension name
    q_dim = "q" if "q" in q_da.dims else "quantile"
    
    # Select nearest quantile slice matching target probability level
    q_slice = q_da.sel({q_dim: eval_quantile}, method="nearest")

    # Determine day and hour dimension names present in slice
    day_dim = (
        "dayofyear"
        if "dayofyear" in q_slice.dims
        else ("day" if "day" in q_slice.dims else None)
    )
    hour_dim = "hour" if "hour" in q_slice.dims else None

    # Return transposed 2D matrix if day and hour dimensions exist
    if day_dim and hour_dim:
        matrix = q_slice.transpose(hour_dim, day_dim).values
        return matrix[:, :365]

    # Tile daily 1D vector across 24 hours if hour dimension is missing
    if day_dim and not hour_dim:
        daily_vec = q_slice.values[:365]
        return np.tile(daily_vec, (24, 1))

    # Return zero array if required temporal dimensions are missing
    return np.zeros((24, 365))


# =====================================================================
# Diagnostic plots part 1 pipeline
# =====================================================================

def plot_qdm_diagnostics(
    ds_obs: Union[xr.DataArray, xr.Dataset],
    ds_ref: Union[xr.DataArray, xr.Dataset],
    ds_qdm_ref: Union[xr.DataArray, xr.Dataset],
    ds_proj: Optional[Union[xr.DataArray, xr.Dataset]] = None,
    ds_qdm_proj: Optional[Union[xr.DataArray, xr.Dataset]] = None,
    diagnostics: Optional[Dict[str, xr.DataArray]] = None,
    figsize: Tuple[int, int] = (16, 20),
) -> Tuple[plt.Figure, np.ndarray]:
    """Plot comprehensive performance diagnostic figures."""
    
    # Standardize observed, historical reference, and adjusted reference inputs to DataArrays
    da_obs = ds_obs.to_array().squeeze() if isinstance(ds_obs, xr.Dataset) else ds_obs
    da_ref = ds_ref.to_array().squeeze() if isinstance(ds_ref, xr.Dataset) else ds_ref
    da_qdm_ref = ds_qdm_ref.to_array().squeeze() if isinstance(ds_qdm_ref, xr.Dataset) else ds_qdm_ref
    
    # Use copy of reference if future projection input is omitted
    if ds_proj is not None:
        da_proj = ds_proj.to_array().squeeze() if isinstance(ds_proj, xr.Dataset) else ds_proj
    else:
        da_proj = da_ref.copy()

    # Use copy of adjusted reference if adjusted future projection is omitted
    if ds_qdm_proj is not None:
        da_qdm_proj = ds_qdm_proj.to_array().squeeze() if isinstance(ds_qdm_proj, xr.Dataset) else ds_qdm_proj
    else:
        da_qdm_proj = da_qdm_ref.copy()

    # Retrieve quantile probability grid from diagnostics or construct default grid
    if diagnostics:
        q_grid = diagnostics["q_grid"]
    else:
        q_grid = np.linspace(0.001, 0.999, 100)

    # Extract unit attribute metadata from reference DataArray
    units_str = da_ref.attrs.get("units", "units")

    # Extract finite numerical values from each dataset
    obs_vals = da_obs.values[~np.isnan(da_obs.values)]
    ref_vals = da_ref.values[~np.isnan(da_ref.values)]
    qdm_ref_vals = da_qdm_ref.values[~np.isnan(da_qdm_ref.values)]
    proj_vals = da_proj.values[~np.isnan(da_proj.values)]
    qdm_proj_vals = da_qdm_proj.values[~np.isnan(da_qdm_proj.values)]

    # Initialize figure canvas and GridSpec layout
    fig = plt.figure(figsize=figsize, constrained_layout=True)
    gs = fig.add_gridspec(4, 2)

    # Panel 1: Empirical Cumulative Distribution Function (CDF)
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(
        np.quantile(obs_vals, q_grid), q_grid, label="Observed",
        color=COLOR_OBS, lw=5.0, ls="-", alpha=0.5
    )
    ax1.plot(
        np.quantile(ref_vals, q_grid), q_grid, label="Raw Reference",
        color=COLOR_RAW_REF, lw=1.5, ls="--"
    )
    ax1.plot(
        np.quantile(qdm_ref_vals, q_grid), q_grid,
        label="QDM Adjusted Reference", color=COLOR_QDM_REF, lw=1.8, ls="-"
    )
    ax1.plot(
        np.quantile(proj_vals, q_grid), q_grid, label="Raw Projection",
        color=COLOR_RAW_PROJ, lw=1.5, ls=":"
    )
    ax1.plot(
        np.quantile(qdm_proj_vals, q_grid), q_grid,
        label="QDM Adjusted Projection", color=COLOR_QDM_PROJ,
        lw=2.0, ls="-"
    )
    ax1.set_xlabel(f"Value ({units_str})")
    ax1.set_ylabel("Quantile Probability")
    ax1.set_title("1. Empirical CDF")
    ax1.legend(frameon=True, loc="upper left", fontsize=8.5)
    ax1.grid(True, alpha=0.3)

    # Panel 2: Quantile-Quantile (Q-Q) Scatter Comparison
    ax2 = fig.add_subplot(gs[0, 1])
    obs_q = np.quantile(obs_vals, q_grid)
    ref_q = np.quantile(ref_vals, q_grid)
    qdm_ref_q = np.quantile(qdm_ref_vals, q_grid)
    proj_q = np.quantile(proj_vals, q_grid)
    qdm_proj_q = np.quantile(qdm_proj_vals, q_grid)

    ax2.plot(
        obs_q, ref_q, label="Raw Reference vs Obs",
        color=COLOR_RAW_REF, lw=1.5, ls="--"
    )
    ax2.plot(
        obs_q, qdm_ref_q, label="QDM Adjusted Ref vs Obs",
        color=COLOR_QDM_REF, lw=1.8, ls="-"
    )
    ax2.plot(
        obs_q, proj_q, label="Raw Proj vs Obs",
        color=COLOR_RAW_PROJ, lw=1.5, ls=":"
    )
    ax2.plot(
        obs_q, qdm_proj_q, label="QDM Adjusted Proj vs Obs",
        color=COLOR_QDM_PROJ, lw=2.0, ls="-"
    )

    # Plot 1:1 reference line spanning data range
    lim_min = min(obs_q.min(), ref_q.min(), qdm_ref_q.min())
    lim_max = max(obs_q.max(), ref_q.max(), qdm_ref_q.max())
    ax2.plot(
        [lim_min, lim_max], [lim_min, lim_max], color="gray",
        linestyle="-.", alpha=0.7, label="1:1 Line"
    )
    ax2.set_xlabel(f"Observed Quantiles ({units_str})")
    ax2.set_ylabel(f"Model Quantiles ({units_str})")
    ax2.set_title("2. Quantile-Quantile (Q-Q) Plot")
    ax2.legend(frameon=True, loc="upper left", fontsize=8.5)
    ax2.grid(True, alpha=0.3)

    # Panel 3: Probability Density Function (PDF) for Reference Period
    ax3 = fig.add_subplot(gs[1, 0])
    v_min_ref = min(obs_vals.min(), ref_vals.min())
    v_max_ref = max(obs_vals.max(), ref_vals.max())
    bins_ref = np.linspace(v_min_ref, v_max_ref, 60)
    
    ax3.hist(
        ref_vals, bins=bins_ref, density=True, alpha=0.25,
        color=COLOR_RAW_REF, edgecolor=COLOR_RAW_REF, linestyle="--",
        label="Raw Reference"
    )
    ax3.hist(
        qdm_ref_vals, bins=bins_ref, density=True, alpha=0.35,
        color=COLOR_QDM_REF, edgecolor=COLOR_QDM_REF, linestyle="-",
        label="QDM Adjusted Reference"
    )
    ax3.hist(
        obs_vals, bins=bins_ref, density=True, histtype="step",
        color=COLOR_OBS, lw=2.0, ls="-", label="Observed Baseline"
    )
    ax3.set_xlabel(f"Value ({units_str})")
    ax3.set_ylabel("Density")
    ax3.set_title("3. Reference Period PDF")
    ax3.legend(frameon=True, loc="upper right", fontsize=8.5)
    ax3.grid(True, alpha=0.3)

    # Panel 4: Probability Density Function (PDF) for Projection Period
    ax4 = fig.add_subplot(gs[1, 1])
    v_min_prj = min(proj_vals.min(), qdm_proj_vals.min())
    v_max_prj = max(proj_vals.max(), qdm_proj_vals.max())
    bins_proj = np.linspace(v_min_prj, v_max_prj, 60)
    
    ax4.hist(
        proj_vals, bins=bins_proj, density=True, alpha=0.25,
        color=COLOR_RAW_PROJ, edgecolor=COLOR_RAW_PROJ, linestyle=":",
        label="Raw Projection"
    )
    ax4.hist(
        qdm_proj_vals, bins=bins_proj, density=True, alpha=0.35,
        color=COLOR_QDM_PROJ, edgecolor=COLOR_QDM_PROJ, linestyle="-",
        label="QDM Adjusted Projection"
    )
    ax4.set_xlabel(f"Value ({units_str})")
    ax4.set_ylabel("Density")
    ax4.set_title("4. Projection Period PDF")
    ax4.legend(frameon=True, loc="upper right", fontsize=8.5)
    ax4.grid(True, alpha=0.3)

    # Panel 5: Seasonal Climatology Grouped by Day of Year
    ax5 = fig.add_subplot(gs[2, :])
    
    # Internal function to calculate 365-day mean climatology
    def _get_daily_mean(da):
        return da.groupby("time.dayofyear").mean("time").values[:365]

    ax5.plot(
        np.arange(1, 366), _get_daily_mean(da_obs), label="Observed",
        color=COLOR_OBS, lw=1.8, ls="-"
    )
    ax5.plot(
        np.arange(1, 366), _get_daily_mean(da_ref), label="Raw Reference",
        color=COLOR_RAW_REF, lw=1.2, ls="--"
    )
    ax5.plot(
        np.arange(1, 366), _get_daily_mean(da_qdm_ref),
        label="QDM Adjusted Reference", color=COLOR_QDM_REF, lw=1.5, ls="-"
    )
    ax5.plot(
        np.arange(1, 366), _get_daily_mean(da_proj),
        label="Raw Projection", color=COLOR_RAW_PROJ, lw=1.2, ls=":"
    )
    ax5.plot(
        np.arange(1, 366), _get_daily_mean(da_qdm_proj),
        label="QDM Adjusted Projection", color=COLOR_QDM_PROJ, lw=1.8,
        ls="-"
    )
    ax5.set_xlabel("Day of Year (1..365)")
    ax5.set_ylabel(f"Mean Value ({units_str})")
    ax5.set_title("5. Day-of-Year Grouped Climatology")
    ax5.legend(frameon=True, loc="upper right", fontsize=8.5)
    ax5.grid(True, alpha=0.3)

    # Panel 6: Sub-daily Diurnal Profiles Grouped by Month in PST
    ax6 = fig.add_subplot(gs[3, :])
    x_c, y_obs, m_c, m_l = _get_monthly_diurnal_cycle_pst(da_obs, pst_offset=-8)
    _, y_ref, _, _ = _get_monthly_diurnal_cycle_pst(da_ref, pst_offset=-8)
    _, y_qdm_ref, _, _ = _get_monthly_diurnal_cycle_pst(da_qdm_ref, pst_offset=-8)
    _, y_proj, _, _ = _get_monthly_diurnal_cycle_pst(da_proj, pst_offset=-8)
    _, y_qdm_proj, _, _ = _get_monthly_diurnal_cycle_pst(da_qdm_proj, pst_offset=-8)

    ax6.plot(
        x_c, y_obs, label="Observed", color=COLOR_OBS, lw=1.8, ls="-"
    )
    ax6.plot(
        x_c, y_ref, label="Raw Reference", color=COLOR_RAW_REF,
        lw=1.2, ls="--"
    )
    ax6.plot(
        x_c, y_qdm_ref, label="QDM Adjusted Reference",
        color=COLOR_QDM_REF, lw=1.5, ls="-"
    )
    ax6.plot(
        x_c, y_proj, label="Raw Projection", color=COLOR_RAW_PROJ,
        lw=1.2, ls=":"
    )
    ax6.plot(
        x_c, y_qdm_proj, label="QDM Adjusted Projection",
        color=COLOR_QDM_PROJ, lw=1.8, ls="-"
    )

    ax6.set_xticks(m_c)
    ax6.set_xticklabels(m_l)
    ax6.set_xlabel("Month (24 Hours / Month, PST)")
    ax6.set_ylabel(f"Mean Value ({units_str})")
    ax6.set_title("6. Monthly Mean Diurnal Cycle (PST)")
    ax6.legend(frameon=True, loc="upper right", fontsize=8.5)
    ax6.grid(True, alpha=0.3)

    # Sub-panel: Inset Zoom Axis for July Diurnal Cycle
    jul_center = m_c[6]
    ax_inset = ax6.inset_axes([0.05, 0.45, 0.28, 0.48])
    ax_inset.plot(x_c, y_obs, color=COLOR_OBS, lw=2.0, ls="-")
    ax_inset.plot(x_c, y_ref, color=COLOR_RAW_REF, lw=1.5, ls="--")
    ax_inset.plot(x_c, y_qdm_ref, color=COLOR_QDM_REF, lw=1.8, ls="-")
    ax_inset.plot(x_c, y_proj, color=COLOR_RAW_PROJ, lw=1.5, ls=":")
    ax_inset.plot(
        x_c, y_qdm_proj, color=COLOR_QDM_PROJ, lw=2.0, ls="-"
    )

    # Restrict inset zoom x-limits to July 24-hour window
    ax_inset.set_xlim(jul_center - 12.0, jul_center + 12.0)
    jul_mask = (x_c >= jul_center - 12.0) & (x_c <= jul_center + 12.0)
    
    # Calculate inset vertical limits across all curves with margin
    y_jul_all = [y_obs[jul_mask], y_ref[jul_mask], y_qdm_ref[jul_mask], y_proj[jul_mask], y_qdm_proj[jul_mask]]
    y_jul_stack = np.concatenate([y[~np.isnan(y)] for y in y_jul_all])
    y_margin = (y_jul_stack.max() - y_jul_stack.min()) * 0.12
    ax_inset.set_ylim(
        y_jul_stack.min() - y_margin, y_jul_stack.max() + y_margin
    )

    # Configure inset tick marks and title
    inset_xt = jul_center - 11.5 + np.array([0, 6, 12, 18, 23])
    ax_inset.set_xticks(inset_xt)
    ax_inset.set_xticklabels(["0", "6", "12", "18", "23"], fontsize=7)
    ax_inset.tick_params(axis="y", labelsize=7)
    ax_inset.set_title("July Zoom (PST)", fontsize=8, pad=3)
    ax_inset.grid(True, alpha=0.3)
    ax6.indicate_inset_zoom(ax_inset, edgecolor="black", alpha=0.45)

    # Construct axes return matrix
    axes = np.array([
        [ax1, ax2], [ax3, ax4], [ax5, ax5], [ax6, ax6]
    ])
    return fig, axes


def plot_qdm_pooled_heatmaps(
    diagnostics: Dict[str, xr.DataArray],
    eval_quantile: float = 0.50,
    units_str: str = "degF",
    figsize: Tuple[int, int] = (16, 20),
) -> Tuple[plt.Figure, np.ndarray]:
    """Plot standalone heatmaps displaying internal QDM pooled quantile matrices,

    including Raw Model Delta, Adjusted Model Delta, and their difference.

    Parameters
    ----------
    diagnostics : dict of {str: xarray.DataArray}
        Pipeline diagnostics dictionary returned by `run_qdm_pipeline`.
    eval_quantile : float, default 0.50
        Quantile probability level (0.0 to 1.0) evaluated across heatmaps.
    units_str : str, default 'degF'
        Units label for plot colorbars.
    figsize : tuple of int, default (16, 20)
        Dimensions (width, height) of the output figure in inches.

    Returns
    -------
    fig : matplotlib.figure.Figure
        Created standalone diagnostic figure instance.
    axes : numpy.ndarray
        Array of subplot axes in a 5x2 grid layout.
    """
    # Extract bias correction mode from pipeline diagnostics
    kind = diagnostics.get("correction_mode", "additive")

    # Step 1: Extract pooled quantile matrices and roll hours to PST
    q_mat_obs = _roll_to_pst(
        _extract_quantile_heatmap(diagnostics, "q_obs", eval_quantile)
    )
    q_mat_ref = _roll_to_pst(
        _extract_quantile_heatmap(diagnostics, "q_ref", eval_quantile)
    )

    # Use projection matrix if present, otherwise copy reference matrix
    # in case we do not want to look at projections
    if "q_proj" in diagnostics:
        q_mat_proj = _roll_to_pst(
            _extract_quantile_heatmap(diagnostics, "q_proj", eval_quantile)
        )
    else:
        q_mat_proj = q_mat_ref.copy()

    # Map adjusted reference quantile matrix directly to observed baseline
    q_mat_qdm_ref = q_mat_obs.copy()

    # Step 2: Derive adjusted projection matrices and delta structures based on correction mode
    if kind == "additive":
        q_mat_qdm_proj = q_mat_obs + (q_mat_proj - q_mat_ref)
        q_ref_factor = q_mat_obs - q_mat_ref
        q_raw_delta = q_mat_proj - q_mat_ref
        q_adj_delta = q_mat_qdm_proj - q_mat_qdm_ref
        q_delta_diff = q_raw_delta - q_adj_delta

        cmap_adj_factor = "PuOr_r"
        label_adj_factor = f"Adjustment Factor ({units_str})"
        cmap_delta = "PuOr_r"
        label_delta = f"Model Delta ({units_str})"
        cmap_diff = "coolwarm"
        label_diff = f"Delta Difference ({units_str})"
    else:
        q_mat_qdm_proj = q_mat_obs * (
            q_mat_proj / np.where(q_mat_ref == 0, 1e-6, q_mat_ref)
        )
        q_ref_factor = q_mat_obs / np.where(
            q_mat_ref == 0, 1e-6, q_mat_ref
        )
        q_raw_delta = q_mat_proj / np.where(
            q_mat_ref == 0, 1e-6, q_mat_ref
        )
        q_adj_delta = q_mat_qdm_proj / np.where(
            q_mat_qdm_ref == 0, 1e-6, q_mat_qdm_ref
        )
        q_delta_diff = q_raw_delta - q_adj_delta

        cmap_adj_factor = "PiYG"
        label_adj_factor = "Adjustment Ratio (Multiplier)"
        cmap_delta = "PiYG"
        label_delta = "Model Delta Ratio (Multiplier)"
        cmap_diff = "coolwarm"
        label_diff = "Ratio Difference"

    # Calculate global value limits across absolute quantile matrices
    vmax_q = max(
        q_mat_ref.max(), q_mat_qdm_ref.max(),
        q_mat_proj.max(), q_mat_qdm_proj.max()
    )
    vmin_q = min(
        q_mat_ref.min(), q_mat_qdm_ref.min(),
        q_mat_proj.min(), q_mat_qdm_proj.min()
    )

    # Step 3: Construct 5x2 subplot layout grid
    fig, axes = plt.subplots(5, 2, figsize=figsize, constrained_layout=True)

    # Panel 1: Observed Pooled Quantiles
    im0 = axes[0, 0].imshow(
        q_mat_obs, aspect="auto", origin="lower", cmap="viridis",
        vmin=vmin_q, vmax=vmax_q, extent=[1, 365, 0, 24]
    )
    fig.colorbar(im0, ax=axes[0, 0], label=f"Value ({units_str})", shrink=0.85)
    axes[0, 0].set_title(f"1. Obs Pooled Quantiles (q = {eval_quantile:.2f})")

    # Panel 2: Adjustment Factor Matrix
    if kind == "additive":
        vmax_adj = max(0.01, np.max(np.abs(q_ref_factor)))
        vmin_adj = -vmax_adj
    else:
        vmax_adj = max(1.01, np.max(q_ref_factor))
        vmin_adj = min(0.99, np.min(q_ref_factor))

    im1 = axes[0, 1].imshow(
        q_ref_factor, aspect="auto", origin="lower", cmap=cmap_adj_factor,
        vmin=vmin_adj, vmax=vmax_adj, extent=[1, 365, 0, 24]
    )
    fig.colorbar(im1, ax=axes[0, 1], label=label_adj_factor, shrink=0.85)
    axes[0, 1].set_title(f"2. QDM Adj Factor Matrix (q = {eval_quantile:.2f})")

    # Panel 3: Raw Reference Pooled Quantiles
    im2 = axes[1, 0].imshow(
        q_mat_ref, aspect="auto", origin="lower", cmap="viridis",
        vmin=vmin_q, vmax=vmax_q, extent=[1, 365, 0, 24]
    )
    fig.colorbar(im2, ax=axes[1, 0], label=f"Value ({units_str})", shrink=0.85)
    axes[1, 0].set_title(f"3. Raw Ref Pooled Quantiles (q = {eval_quantile:.2f})")

    # Panel 4: QDM Adjusted Reference Pooled Quantiles
    im3 = axes[1, 1].imshow(
        q_mat_qdm_ref, aspect="auto", origin="lower", cmap="viridis",
        vmin=vmin_q, vmax=vmax_q, extent=[1, 365, 0, 24]
    )
    fig.colorbar(im3, ax=axes[1, 1], label=f"Value ({units_str})", shrink=0.85)
    axes[1, 1].set_title(f"4. QDM Adj Ref Pooled Quantiles (q = {eval_quantile:.2f})")

    # Panel 5: Raw Projection Pooled Quantiles
    im4 = axes[2, 0].imshow(
        q_mat_proj, aspect="auto", origin="lower", cmap="viridis",
        vmin=vmin_q, vmax=vmax_q, extent=[1, 365, 0, 24]
    )
    fig.colorbar(im4, ax=axes[2, 0], label=f"Value ({units_str})", shrink=0.85)
    axes[2, 0].set_title(f"5. Raw Proj Pooled Quantiles (q = {eval_quantile:.2f})")

    # Panel 6: QDM Adjusted Projection Pooled Quantiles
    im5 = axes[2, 1].imshow(
        q_mat_qdm_proj, aspect="auto", origin="lower", cmap="viridis",
        vmin=vmin_q, vmax=vmax_q, extent=[1, 365, 0, 24]
    )
    fig.colorbar(im5, ax=axes[2, 1], label=f"Value ({units_str})", shrink=0.85)
    axes[2, 1].set_title(f"6. QDM Adj Proj Pooled Quantiles (q = {eval_quantile:.2f})")

    # Calculate symmetric bounds for delta matrices
    if kind == "additive":
        vmax_delta = max(0.01, np.max(np.abs(q_raw_delta)), np.max(np.abs(q_adj_delta)))
        vmin_delta = -vmax_delta
    else:
        vmax_delta = max(1.01, np.max(q_raw_delta), np.max(q_adj_delta))
        vmin_delta = min(0.99, np.min(q_raw_delta), np.min(q_adj_delta))

    # Panel 7: Raw Model Delta Matrix (Proj - Ref)
    im6 = axes[3, 0].imshow(
        q_raw_delta, aspect="auto", origin="lower", cmap=cmap_delta,
        vmin=vmin_delta, vmax=vmax_delta, extent=[1, 365, 0, 24]
    )
    fig.colorbar(im6, ax=axes[3, 0], label=label_delta, shrink=0.85)
    axes[3, 0].set_title(f"7. Raw Model Delta Matrix (q = {eval_quantile:.2f})")

    # Panel 8: QDM Adjusted Model Delta Matrix (QDM Proj - QDM Ref)
    im7 = axes[3, 1].imshow(
        q_adj_delta, aspect="auto", origin="lower", cmap=cmap_delta,
        vmin=vmin_delta, vmax=vmax_delta, extent=[1, 365, 0, 24]
    )
    fig.colorbar(im7, ax=axes[3, 1], label=label_delta, shrink=0.85)
    axes[3, 1].set_title(f"8. QDM Adj Model Delta Matrix (q = {eval_quantile:.2f})")

    # Panel 9: Delta Difference Matrix (Raw Delta - Adjusted Delta)
    vmax_diff = max(0.01, np.max(np.abs(q_delta_diff)))
    vmin_diff = -vmax_diff

    im8 = axes[4, 0].imshow(
        q_delta_diff, aspect="auto", origin="lower", cmap=cmap_diff,
        vmin=vmin_diff, vmax=vmax_diff, extent=[1, 365, 0, 24]
    )
    fig.colorbar(im8, ax=axes[4, 0], label=label_diff, shrink=0.85)
    axes[4, 0].set_title(f"9. Delta Difference (Raw Delta - Adj Delta, q = {eval_quantile:.2f})")

    # Hide unused subplot cell in 5x2 grid layout
    axes[4, 1].axis("off")

    # Apply x and y axis labels to all active heatmaps
    for ax in axes.flat[:-1]:
        ax.set_xlabel("Day of Year")
        ax.set_ylabel("Hour of Day (PST)")

    return fig, axes

    
# =====================================================================
# Diagnostic plots part 2: QDM windowing intercomparison
# =====================================================================


# Palette for Delta / Sensitivity Analysis
COLOR_OBS = "#000000"       # Black (Observed Baseline)
COLOR_RAW = "#555555"       # Dark Gray (Raw Model Baseline)
COLOR_PIPE_A = "#D55E00"    # Vermillion/Orange (Pipeline A: e.g., Daily QDM)
COLOR_PIPE_B = "#0072B2"    # Deep Blue (Pipeline B: e.g., Hourly QDM)


def _extract_quantile_heatmap(
    diagnostics: Dict[str, xr.DataArray],
    key: str,
    eval_quantile: float = 0.50,
) -> np.ndarray:
    """Extract a (24, 365) matrix for a given quantile key from diagnostics.

    Parameters
    ----------
    diagnostics : dict of {str: xarray.DataArray}
        Pipeline diagnostics dictionary returned by `run_qdm_pipeline`.
    key : str
        Target diagnostic dictionary key (e.g., 'q_obs', 'q_ref', 'q_proj').
    eval_quantile : float, default 0.50
        Quantile level (0.0 to 1.0) to evaluate.

    Returns
    -------
    numpy.ndarray
        2D matrix of shape (24, 365).
    """
    # Retrieve target DataArray from diagnostics dictionary
    q_da = diagnostics[key]
    
    # Identify quantile dimension name
    q_dim = "q" if "q" in q_da.dims else "quantile"
    
    # Select nearest quantile slice matching target probability level
    q_slice = q_da.sel({q_dim: eval_quantile}, method="nearest")

    # Determine day and hour dimension names present in slice
    day_dim = (
        "dayofyear"
        if "dayofyear" in q_slice.dims
        else ("day" if "day" in q_slice.dims else None)
    )
    hour_dim = "hour" if "hour" in q_slice.dims else None

    # Hour windowing ENABLED: Extract 2D diurnal x seasonal matrix transposed to (hour, dayofyear)
    if day_dim and hour_dim:
        matrix = q_slice.transpose(hour_dim, day_dim).values
        return matrix[:, :365]

    # Hour windowing DISABLED: Tile 1D daily vector across all 24 hours
    if day_dim and not hour_dim:
        daily_vec = q_slice.values[:365]
        return np.tile(daily_vec, (24, 1))

    # Return zero array if required temporal dimensions are missing
    return np.zeros((24, 365))

    
# =====================================================================
# Diagnostic plots part 2 pipeline
# =====================================================================

    
def plot_qdm_sensitivity_deltas(
    ds_obs: xr.DataArray,
    ds_ref_raw: xr.DataArray,
    ds_qdm_ref_a: xr.DataArray,
    ds_qdm_ref_b: xr.DataArray,
    diag_a: Dict[str, xr.DataArray],
    diag_b: Dict[str, xr.DataArray],
    label_a: str = "Daily QDM",
    label_b: str = "Hourly QDM",
    ds_proj_raw: Optional[xr.DataArray] = None,
    ds_qdm_proj_a: Optional[xr.DataArray] = None,
    ds_qdm_proj_b: Optional[xr.DataArray] = None,
    eval_quantile: float = 0.50,
    pst_offset: int = -8,
    figsize: Optional[Tuple[int, int]] = (16, 26),
) -> Tuple[plt.Figure, np.ndarray]:
    """Plot sensitivity and pipeline delta diagnostic suite comparing two pipelines.

    Parameters
    ----------
    ds_obs : xarray.DataArray
        Observed historical baseline dataset.
    ds_ref_raw : xarray.DataArray
        Raw historical model reference dataset.
    ds_qdm_ref_a : xarray.DataArray
        QDM-adjusted reference time series from Pipeline A.
    ds_qdm_ref_b : xarray.DataArray
        QDM-adjusted reference time series from Pipeline B.
    diag_a : dict of {str: xarray.DataArray}
        Diagnostics dictionary from Pipeline A.
    diag_b : dict of {str: xarray.DataArray}
        Diagnostics dictionary from Pipeline B.
    label_a : str, default "Daily QDM"
        Display label for Pipeline A.
    label_b : str, default "Hourly QDM"
        Display label for Pipeline B.
    ds_proj_raw : xarray.DataArray, optional
        Raw projection dataset.
    ds_qdm_proj_a : xarray.DataArray, optional
        QDM-adjusted projection time series from Pipeline A.
    ds_qdm_proj_b : xarray.DataArray, optional
        QDM-adjusted projection time series from Pipeline B.
    eval_quantile : float, default 0.50
        Quantile level evaluated across heatmaps.
    pst_offset : int, default -8
        Hour offset to convert UTC to PST.
    figsize : tuple of int, optional, default (16, 26)
        Dimensions of the output figure in inches.

    Returns
    -------
    fig : matplotlib.figure.Figure
        Created sensitivity diagnostic figure.
    axes : numpy.ndarray
        Array of subplot axes in a 6x2 grid layout.
    """
    # Extract unit attribute metadata and correction mode
    units_str = ds_ref_raw.attrs.get("units", "units")
    kind = diag_a.get("correction_mode", "additive") if diag_a else "additive"

    # Initialize figure canvas and GridSpec layout
    fig = plt.figure(figsize=figsize, constrained_layout=True)
    gs = gridspec.GridSpec(6, 2, figure=fig)

    # -------------------------------------------------------------------------
    # Row 1: PDF Density Deltas (Panels 1 & 2)
    # -------------------------------------------------------------------------
    # Extract finite numerical values for reference period series
    ref_a_vals = ds_qdm_ref_a.values[~np.isnan(ds_qdm_ref_a.values)]
    ref_b_vals = ds_qdm_ref_b.values[~np.isnan(ds_qdm_ref_b.values)]

    # Panel 1: Reference Period PDF Density Delta Histogram
    ax1 = fig.add_subplot(gs[0, 0])
    bins_ref = np.linspace(
        min(ref_a_vals.min(), ref_b_vals.min()),
        max(ref_a_vals.max(), ref_b_vals.max()),
        50,
    )
    counts_a_ref, _ = np.histogram(ref_a_vals, bins=bins_ref, density=True)
    counts_b_ref, _ = np.histogram(ref_b_vals, bins=bins_ref, density=True)
    delta_pdf_ref = counts_b_ref - counts_a_ref

    bin_centers_ref = (bins_ref[:-1] + bins_ref[1:]) / 2.0
    bin_width_ref = bins_ref[1] - bins_ref[0]

    # Assign bar colors according to sign of density delta
    colors_bar_ref = np.where(delta_pdf_ref >= 0, COLOR_PIPE_B, COLOR_PIPE_A)
    ax1.bar(
        bin_centers_ref,
        delta_pdf_ref,
        width=bin_width_ref * 0.9,
        color=colors_bar_ref,
        alpha=0.75,
        edgecolor="black",
        linewidth=0.5,
    )
    ax1.axhline(0, color="black", linestyle="--", alpha=0.7, lw=1.0)
    ax1.set_xlabel(f"Value ({units_str})")
    ax1.set_ylabel(r"$\Delta$ Density Residual")
    ax1.set_title(f"1. Ref PDF Density Delta ({label_b} - {label_a})")

    legend_patches_ref = [
        mpatches.Patch(
            facecolor=COLOR_PIPE_B,
            alpha=0.75,
            edgecolor="black",
            label=f"Higher in {label_b}",
        ),
        mpatches.Patch(
            facecolor=COLOR_PIPE_A,
            alpha=0.75,
            edgecolor="black",
            label=f"Higher in {label_a}",
        ),
    ]
    ax1.legend(
        handles=legend_patches_ref,
        frameon=True,
        loc="upper right",
        fontsize=8.5,
    )
    ax1.grid(True, alpha=0.3)

    # Panel 2: Projection Period PDF Density Delta Histogram
    ax2 = fig.add_subplot(gs[0, 1])
    delta_pdf_proj = np.array([0.0])

    if ds_qdm_proj_a is not None and ds_qdm_proj_b is not None:
        proj_a_vals = ds_qdm_proj_a.values[~np.isnan(ds_qdm_proj_a.values)]
        proj_b_vals = ds_qdm_proj_b.values[~np.isnan(ds_qdm_proj_b.values)]

        bins_proj = np.linspace(
            min(proj_a_vals.min(), proj_b_vals.min()),
            max(proj_a_vals.max(), proj_b_vals.max()),
            50,
        )
        counts_a_proj, _ = np.histogram(
            proj_a_vals, bins=bins_proj, density=True
        )
        counts_b_proj, _ = np.histogram(
            proj_b_vals, bins=bins_proj, density=True
        )
        delta_pdf_proj = counts_b_proj - counts_a_proj

        bin_centers_proj = (bins_proj[:-1] + bins_proj[1:]) / 2.0
        bin_width_proj = bins_proj[1] - bins_proj[0]

        colors_bar_proj = np.where(
            delta_pdf_proj >= 0, COLOR_PIPE_B, COLOR_PIPE_A
        )
        ax2.bar(
            bin_centers_proj,
            delta_pdf_proj,
            width=bin_width_proj * 0.9,
            color=colors_bar_proj,
            alpha=0.75,
            edgecolor="black",
            linewidth=0.5,
        )
        ax2.axhline(0, color="black", linestyle="--", alpha=0.7, lw=1.0)
        ax2.set_xlabel(f"Value ({units_str})")
        ax2.set_ylabel(r"$\Delta$ Density Residual")
        ax2.set_title(f"2. Proj PDF Density Delta ({label_b} - {label_a})")

        legend_patches_proj = [
            mpatches.Patch(
                facecolor=COLOR_PIPE_B,
                alpha=0.75,
                edgecolor="black",
                label=f"Higher in {label_b}",
            ),
            mpatches.Patch(
                facecolor=COLOR_PIPE_A,
                alpha=0.75,
                edgecolor="black",
                label=f"Higher in {label_a}",
            ),
        ]
        ax2.legend(
            handles=legend_patches_proj,
            frameon=True,
            loc="upper right",
            fontsize=8.5,
        )

    ax2.grid(True, alpha=0.3)

    # Harmonize vertical limits across both PDF residual subplots
    y_max_pdf = max(np.max(delta_pdf_ref), np.max(delta_pdf_proj))
    y_min_pdf = min(np.min(delta_pdf_ref), np.min(delta_pdf_proj))
    pdf_range_pad = (
        (y_max_pdf - y_min_pdf) * 0.10 if y_max_pdf != y_min_pdf else 0.01
    )
    ax1.set_ylim(y_min_pdf - pdf_range_pad, y_max_pdf + pdf_range_pad)
    ax2.set_ylim(y_min_pdf - pdf_range_pad, y_max_pdf + pdf_range_pad)

    # -------------------------------------------------------------------------
    # Row 2: Day-of-Year Mean Residual Delta (Panel 3)
    # -------------------------------------------------------------------------
    ax3 = fig.add_subplot(gs[1, :])

    # Internal function to calculate 365-day mean climatology
    def _get_daily_mean(da):
        return da.groupby("time.dayofyear").mean("time").values[:365]

    mean_a_ref = _get_daily_mean(ds_qdm_ref_a)
    mean_b_ref = _get_daily_mean(ds_qdm_ref_b)
    ax3.plot(
        np.arange(1, 366),
        mean_b_ref - mean_a_ref,
        color=COLOR_PIPE_A,
        lw=2.2,
        label=f"Ref Delta ({label_b} - {label_a})",
    )

    if ds_qdm_proj_a is not None and ds_qdm_proj_b is not None:
        mean_a_proj = _get_daily_mean(ds_qdm_proj_a)
        mean_b_proj = _get_daily_mean(ds_qdm_proj_b)
        ax3.plot(
            np.arange(1, 366),
            mean_b_proj - mean_a_proj,
            color=COLOR_PIPE_B,
            lw=2.2,
            ls="--",
            label=f"Proj Delta ({label_b} - {label_a})",
        )

    ax3.axhline(0, color="black", linestyle=":", alpha=0.6)
    ax3.set_xlabel("Day of Year (1..365)")
    ax3.set_ylabel(f"$\Delta$ Mean Value ({units_str})")
    ax3.set_title(
        f"3. Day-of-Year Mean Residual Delta ({label_b} - {label_a})"
    )
    ax3.legend(frameon=True, loc="upper right", fontsize=8.5)
    ax3.grid(True, alpha=0.3)

    # -------------------------------------------------------------------------
    # Row 3: Month-of-Year Diurnal Cycle Residual Delta (Panel 4)
    # -------------------------------------------------------------------------
    ax4 = fig.add_subplot(gs[2, :])

    # Extract monthly diurnal profiles shifted to PST for both pipelines
    x_coords, diurnal_a_ref, month_centers, month_labels = (
        _get_monthly_diurnal_cycle_pst(ds_qdm_ref_a, pst_offset=pst_offset)
    )
    _, diurnal_b_ref, _, _ = _get_monthly_diurnal_cycle_pst(
        ds_qdm_ref_b, pst_offset=pst_offset
    )

    ax4.plot(
        x_coords,
        diurnal_b_ref - diurnal_a_ref,
        color=COLOR_PIPE_A,
        lw=2.0,
        label=f"Ref Diurnal Delta ({label_b} - {label_a})",
    )

    if ds_qdm_proj_a is not None and ds_qdm_proj_b is not None:
        _, diurnal_a_proj, _, _ = _get_monthly_diurnal_cycle_pst(
            ds_qdm_proj_a, pst_offset=pst_offset
        )
        _, diurnal_b_proj, _, _ = _get_monthly_diurnal_cycle_pst(
            ds_qdm_proj_b, pst_offset=pst_offset
        )
        ax4.plot(
            x_coords,
            diurnal_b_proj - diurnal_a_proj,
            color=COLOR_PIPE_B,
            lw=2.0,
            ls="--",
            label=f"Proj Diurnal Delta ({label_b} - {label_a})",
        )

    ax4.set_xticks(month_centers)
    ax4.set_xticklabels(month_labels)
    ax4.axhline(0, color="black", linestyle=":", alpha=0.6)
    ax4.set_xlabel("Month (24 Hours / Month, PST)")
    ax4.set_ylabel(f"$\Delta$ Mean Value ({units_str})")
    ax4.set_title(
        f"4. Month-of-Year Diurnal Cycle Mean Residual Delta "
        f"({label_b} - {label_a})"
    )
    ax4.legend(frameon=True, loc="upper right", fontsize=8.5)
    ax4.grid(True, alpha=0.3)

    # Construct axes array layout
    axes = np.array(
        [[ax1, ax2], [ax3, ax3], [ax4, ax4]]
    )
    
    return fig, axes

In [ ]:
obs_ds = xr.open_dataset('ksan_smoothed_tas/',engine="zarr")
obs_ds = obs_ds.tas.compute()
display(obs_ds)

In [ ]:
cd = ck.ClimateData(verbosity=-2)
time_slice = (1981, 2075)
wrf_ds = (
    cd.catalog("cadcat")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id("1hr")
    .grid_label("d03")
    .variable("t2")
    .processes({
        "clip": "KSAN",
        "convert_units": "degF",
        "time_slice": time_slice
    })
    .get()
)
display(wrf_ds)

In [ ]:
# All of my comments throughout are based on the following simulation
# but you can pick a different one if you want

wrf_ds = wrf_ds.sel(sim = 'wrf_ucla_ec-earth3-veg_ssp370_r1i1p1f1').squeeze()
# wrf_ds = wrf_ds.sel(sim = 'wrf_ucla_taiesm1_ssp370_r1i1p1f1').squeeze()

wrf_ds = wrf_ds["t2"]
drop_list = list(c for c in wrf_ds.coords if c!="time")
wrf_ds = wrf_ds.convert_calendar("noleap")
wrf_ds = wrf_ds.drop_vars(drop_list)

hist_ds = wrf_ds.sel(
        time=slice(str(obs_ds.time.values[0]),
        str(obs_ds.time.values[-1]))).compute()
sim_ds = wrf_ds.sel(
    time=slice("2046-01-01 00:00:00",
               "2075-12-31 23:00:00")).compute()

# Rolling window sensitivity tests

The localization procedure works by sampling over a given time window to build a distribution over which the quantiles and adjustment factors are calculated. Here we compare results for different rolling windows: day-of-year, which pools all hours in a rolling 61-day window together to perform a daily QDM which accounts for seasonality, and hour-of-year, which pools rolling 3-hour windows taken over rolling 61-day windows together to perform an hourly QDM which accounts for seasonality *and* diurnality simultaneously. 

**Hypothesis: An hour-of-year procedure which accounts for both seasonal and diurnal background variability will result in localized results which more faithfully capture the observation's diurnal signal than a day-of-year procedure.** 

# Part 1: Day-of-year QDM (Seasonally aware only)

Compute statistics and adjustment factors using rolling daily windows. 

As a reminder, **Day-of-Year Windowing** involves using a multi-day window (e.g., 61 days) centered on a target day of the year and pools neighboring days across all available historical years. For example, the sample representing May 1st would involve pooling all hours from April 1st to June 1st (approximately, idk I didn't actually count the days but you get what I mean right?).

In [ ]:
%%time

# Run QDM
# Start with day-of-year (pool all hours in the rolling day windows to build the sample)
ds_qdm_proj_doy, ds_qdm_ref_doy, diagnostics_doy = run_qdm_pipeline(
    ds_obs=obs_ds,
    ds_ref=hist_ds,
    ds_proj=sim_ds,
    hour_window=None,
    day_window=61,
    return_intermediate=True,
)

## Figure 1: The statistics which the day-of-year QDM "sees"

We show here that the day-of-year (`hour_window=None`) localization works as expected by computing **daily** statistics and subsequent adjustment factors. Each day's sample is built by pooling all values within a 61-day rolling window to maintain **seasonal variability** without accounting for the diurnal cycle. Accordingly, there is no diurnal variation shown in these heatmaps --- the 95th percentile shown here corresponds to that day's 95th percentile as conditioned within that +/- 30 day window, the adjustment factors obtained via the localization procedure, and the results after applying those daily adjustment factors. 


In [ ]:
# Under-the-hood pooled heatmaps
fig_under_hood_doy, axes_under_hood_doy = plot_qdm_pooled_heatmaps(
    diagnostics_doy, eval_quantile=0.95, units_str="degF"
)

### Figures 1.1, 1.3, and 1.4: Extreme Quantiles Baseline

* **Figure 1.1: Observed Baseline:** Peak extreme temperatures ($q = 0.95$) occur during late summer (Days 200–270).


* **Figure 1.3: Raw Reference Model Bias:** The raw model overestimates extreme heat during summer months compared to observations, expanding the high-temperature zone (yellow region).


* **Figure 1.4: QDM Bias Correction:** Applying QDM scales back the summer high-temperature extremes, aligning the adjusted reference heatmap (Panel 4) into with the observed baseline (1.1).



---

### Figure 1.2. Adjustment Factor Matrix

* **QDM Adj Factor Matrix = Obs - Raw Ref:** Shows the adjustment factors applied across all 365 days and 24 hours. There is a strong cooling correction during summer peak heat (Days 180–280), illustrated by dark purple bands which show negative adjustment factors down to -6.0 to -7.0 °F to suppress the model's warm bias.


---

### Figures 1.5, 1.6: Future Projections

* **Raw vs. Adjusted Future Projection:** The raw future projection (Figure 1.5) predicts widespread extreme temperatures exceeding 87.5 °F across summer. The QDM-adjusted projection (Figure 1.6) tempers this extreme bias while preserving the projected warming signal.


---

### Figures 1.7 and 1.8: Model Delta Matrices

* **Figure 1.7 (Raw Projection - Raw Reference)** The projected climate change delta across the year in the **raw** model output.

  
* **Figure 1.8 (Adjusted Projection - Adjusted Reference)** The projected climate change delta across the year in the **QDM adjusted** model output. 


All values are positive (orange/brown shading), indicating universal warming across all days and hours.


---

### Figure 1.9: Raw Delta - Adjusted Delta

* **Raw Delta - QDM Adj Delta:** The matrix is uniformly zero across all 365 days and 24 hours. This shows that the QDM bias correction is delta-preserving in additive mode, ie, it corrects historical model baseline errors without distorting or altering the model's underlying climate change signal.


---

### Key Takeaway

For extreme $q = 0.95$ heat events, the raw model exhibits a significant warm bias in summer. QDM successfully applies a strong negative adjustment factor (up to $-7\ ^\circ\text{F}$) during summer months while preserving a consistent projected warming delta of $+3.5\text{ to }+4.5\ ^\circ\text{F}$ in the future climate scenario.

---

## **Figure 2: Day-of-year Adjustment Statistics**

The following plots illustrate some basic results of the day-of-year QDM procedure. Descriptions and discussion are below the plots.

In [ ]:
# Plot diagnostic suite
fig, axes = plot_qdm_diagnostics(
    ds_obs=obs_ds,
    ds_ref=hist_ds,
    ds_qdm_ref=ds_qdm_ref_doy,
    ds_proj=sim_ds,
    ds_qdm_proj=ds_qdm_proj_doy,
    diagnostics=diagnostics_doy,
)

#### Figure 2.1 Empirical CDF & Figure 2.2 Q-Q Plot (Top row)

* **Baseline Correction (Orange/Yellow):** The raw reference model (dashed orange) exhibits a cold bias relative to observations (thick black line) until approximately the 70th quantile, beyond which the model runs hot. After applying QDM (solid orange), the adjusted reference distribution aligns with the observed baseline CDF and follows the 1:1 line across all quantile levels.


* **Future Projection (Blue/Green):** The raw projection (dotted blue) is shifted to higher temperatures. Applying QDM (solid green) preserves the modeled climate delta while removing baseline quantile biases relative to the observed record.



---

#### Figure 2.3 Reference Period PDF & Figure 2.4 Projection Period PDF (Second row)

* **Figure 2.3 (Reference Period):** Highlights the bias correction in historical model outputs. The leftward skew and rightward tail are adjusted to a symmetrical Gaussian distribution. The QDM-adjusted reference distribution (yellow bars) closely matches the empirical histogram of observed baseline temperatures (black step outline).


* **Figure 2.4 (Projection Period):** Shows the future projected distribution before (light blue bars) and after (green bars) bias adjustment, preserving the overall variance while shifting peak probabilities.



---

#### Figure 2.5 Day-of-Year Grouped Climatology (Third row)

Shows daily mean temperatures across the 365-day seasonal cycle.


* **Baseline Alignment:** Raw reference data (dashed orange) underestimates daily means in the spring and overestimates them in the summer by ~1–2 °F. QDM adjustment (solid orange) aligns the reference curve with the observed daily means (black line).


* **Future Warming:** The QDM-adjusted future projection (solid green) preserves the seasonal warming profile of the raw model projection (dotted blue), maintaining consistent relative seasonal shifts. In particular, it decreases summertime mean temperatures, **an adjustment which can have heavy implications for the energy sector.**



---

#### Figure 2.6. Monthly Mean Diurnal Cycle (Fourth row)

Evaluates intra-day 24-hour temperature dynamics across each month of the year (in PST).


* Raw model data (dashed orange / dotted blue) underpredicts nocturnal (sunset -- sunrise) temperatures and overpredicts peak afternoon max temperatures.


* QDM adjustment (solid lines) more closely aligns the model output with the observational highs and lows, though there are still marked biases: **nocturnal lows are still too low, and diurnal highs are still too high.**


* **July Inset Detail:** Provides a closer look at the original and remaining biases, which are +/- ~5 °F and +/- ~1 °F, respectively. The biases in the mean are minimized now, but the mean does not tell the whole story.



---

#### Key Takeaway

The QDM pipeline corrects systematic biases present in the raw model baseline while preserving the projected future climate warming signals across **daily** and **seasonal** time scales. That said, there still exist overnight cold biases and daytime warm biases. Can we do better?

# Part 2: Combined day-of-year and hour-of-day ("hour-of-year") QDM (Seasonally and diurnally aware)

Compute statistics and adjustment factors using rolling daily and hourly windows.

As a reminder, **Hour-of-Year Windowing** involves taking a multi-hour window (e.g., 3 hours) centered on a target hour of day and pools neighboring hours around the target time alongside the multi-day window, capturing both seasonal and diurnal variability. For example, the sample for May 1st at 12:00 would be drawn by pooling all measurements between April 1 and June 1 **for the hours spanning 11:00 to 13:00 only**, rather than all hours of the day as done in the previous example.

In [ ]:
%%time

# Run QDM
# Hour-of-year (rolling hourly windows)
ds_qdm_proj_hoy, ds_qdm_ref_hoy, diagnostics_hoy = run_qdm_pipeline(
    ds_obs=obs_ds,
    ds_ref=hist_ds,
    ds_proj=sim_ds,
    hour_window=3,
    day_window=61,
    return_intermediate=True,
)


---

## **Figure 3: The hour-of-year statistics which the QDM "sees"**

We show here that the hour-of-year (here `hour_window=3`) localization works as expected by computing hourly statistics and subsequent adjustment factors. Each hour's sample is built by pooling all values within a 3-hour, 61-day rolling window to maintain diurnal and seasonal variability. Accordingly, there is diurnal variation shown in these heatmaps --- the 95th percentile shown here corresponds to that hour's 95th percentile as sampled within that +/- 1 hour, +/- 30 day window, as well as the adjustment factor obtained via the localization procedure, and the results after applying those hourly adjustment factors. 

In [ ]:
# Under-the-hood pooled heatmaps
fig_under_hood_hoy, axes_under_hood_hoy = plot_qdm_pooled_heatmaps(
    diagnostics_hoy, eval_quantile=0.95, units_str="degF"
)

---

### Figures 3.1, 3.3, and 3.4: Extreme Quantiles Baseline

* **Observed Baseline (Figure 3.1):** Peak extreme temperatures ($q = 0.95$) reach 85.0–90.0 °F during summer afternoons (Days 200–270, Hours 12–16 PST).
* **Raw Reference Model Bias (Figure 3.3):** The raw model overestimates extreme heat during summer afternoon hours, creating an exaggerated yellow core exceeding 90.0 °F.
* **QDM Bias Adjustment (Figure 3.4):** QDM scales back these excessive afternoon extremes, aligning the adjusted reference heatmap in both seasonal and diurnal structure with the observed baseline (Figure 3.1).

---

### Figure 3.2: Adjustment Factor Matrix

* **QDM Adj Factor Matrix = Obs - Raw Ref:** Displays strong diurnal contrast rather than uniform vertical bands (as in Figure 2.2).
* **Midday/Afternoon Cooling (Hours 10–16 PST):** A dark purple horizontal band shows strong negative adjustments (-5.0 to -7.5 °F) required to curb midday model overheating.
* **Night/Morning Warming (Hours 0–8 PST & 18–23 PST):** Light orange shading indicates slight positive adjustments (+2.5 to +5.0 °F), correcting an early morning cold bias.



---

### Figures 3.5 and 3.6: Future Projections

* **Raw vs. Adjusted Future Projection:** The raw future projection (Figure 3.5) predicts prolonged afternoon extreme heat exceeding 95.0 °F. The QDM-adjusted projection (Figure 3.6) tempers these midday peaks while preserving the projected climate warming signal.

---

### Figures 3.7 and 3.8: Model Delta Matrices

* **Figure 3.7 (Raw Model Delta = Raw Proj - Raw Ref):** The projected climate change warming signal across the year and diurnal cycle in the raw model output.
* **Figure 3.8 (QDM Adj Model Delta = QDM Proj - QDM Ref):** The projected climate change warming signal in the QDM-adjusted model output.

Both figures demonstrate universal warming across all days and hours, with peak warming (**+4.0 to +6.0 °F**) concentrated during late summer and autumn afternoons.

---

### Figure 3.9: Delta Difference (Raw Delta - Adjusted Delta)

* **Figure 3.9 (Raw Delta - QDM Adj Delta):** The matrix is uniformly zero across all 365 days and 24 hours. This shows that the QDM bias correction is delta-preserving in additive mode, ie, it corrects historical model baseline errors without distorting or altering the model's underlying climate change signal.

---

### Key Takeaway

For extreme $q = 0.95$ heat events, the raw model exhibits a strong diurnal bias in which it overestimates midday peak temperatures and underestimates nighttime minimua. QDM successfully applies a targeted afternoon cooling adjustment (up to $-7.5\ ^\circ\text{F}$) while preserving the model's projected climate delta of $+4.0\text{ to }+6.0\ ^\circ\text{F}$ across all hours of the year.

---

## **Figure 4: Hour-of-year Adjustment Statistics**

The following plots illustrate some basic results of the hour-of-year QDM procedure. Descriptions and discussion are below the plots.

In [ ]:
# Plot diagnostic suite
fig, axes = plot_qdm_diagnostics(
    ds_obs=obs_ds,
    ds_ref=hist_ds,
    ds_qdm_ref=ds_qdm_ref_hoy,
    ds_proj=sim_ds,
    ds_qdm_proj=ds_qdm_proj_hoy,
    diagnostics=diagnostics_hoy,
)

Most of the Figures 4.1--4.5 are similar enough to Figures 2.1--2.5 that we won't discuss them. The big change appears in Figure 4.6.

---

### Figure 4.6: Monthly Mean Diurnal Cycle (PST)

Figure 2.6 showed improvement in the diurnal cycle's amplitude, but biases still existed: nights were still too cold and days were still too warm. By contrast, the **hour-of-year QDM successfully adjusts the trough and peak amplitudes to be in line with observations**. 

**Why does this matter?** While the initial improvements in the daily mean (Figures 2.5 and 4.5) already have implications from an energy standpoint (eg, the original hot bias would imply substantially more cooling demand than the adjusted data), the diurnal cycle matters too. Temperatures around sunrise in the cooler seasons drive peak heating demand and temperatures around sunset in the warm season drive peak cooling demands. Both of these times suffer from a lack of PV generation, taking out not only a significant portion of utility energy generation but also the behind-the-meter production (ie, rooftop PV) whose contributions can lessen peak load on the grid. That said, the increase in mid-day behind-the-meter PV production over the past several years also **steepens the net load ramp** (called the ["duck curve"](https://www.eia.gov/todayinenergy/detail.php?id=56880)), making it challenging for grid operators to balance supply and demand as they have to more quickly ramp up power generation from other resources (eg, natural gas). **The overall flattening of the diurnal temperature cycle by the hour-of-year adjustment procedure is actually very reassuring from this perspective**, and is a finding which would have been lost had we left the procedure with the day-of-year windowing alone. 



---

### Key Takeaway

By pooling quantiles over a combined day-of-year and hour-of-day window (called here "hour-of-year" windowing), QDM simultaneously resolves **seasonal mean biases** and **sub-daily diurnal amplitude distortions** present in the raw model, all while preserving the model's underlying climate projection delta. Overall our hypothesis that hour-of-year windowing would result in a more consistent diurnal cycle with respect to observations is supported by these results. 

---

# Intercomparison diagnostics

## **Figure 5: Residuals**
Let's check out some differences. In all, residuals between the hour-of-year and day-of-year adjusted data are quite small at the PDF and daily mean level, but the diurnal cycle does show the key differences between them. Specifically, Figure 5.4 reflects the more positive nighttime and more negative daytime adjustments performed by the hour-of-year windowing relative to the day-of-year windowing. 

In [ ]:
fig, axes = plot_qdm_sensitivity_deltas(
    ds_obs=obs_ds,
    ds_ref_raw=hist_ds,
    ds_qdm_ref_a=ds_qdm_ref_doy,
    ds_qdm_ref_b=ds_qdm_ref_hoy,
    diag_a=diagnostics_doy,
    diag_b=diagnostics_hoy,
    label_a="Daily QDM",
    label_b="Hourly QDM",
    ds_proj_raw=sim_ds,
    ds_qdm_proj_a=ds_qdm_proj_doy,
    ds_qdm_proj_b=ds_qdm_proj_hoy,
    eval_quantile=0.95,
)

fig.suptitle(
    "Sensitivity Test: Daily (Seasonal) QDM vs. Hourly (Seasonal + Diurnal) QDM",
    fontsize=16,
    y=1.01,
)

# Yay you're done!

But if you think of other metrics or plots to evaluate results, I would love to see them :) Or just tell me what you're thinking!